# Day 21 — Streaming Lab: End-to-End Pipeline

**What you will learn:**
- Build a complete streaming pipeline from raw events to Bronze → Silver → Gold
- Stream + batch join: enriching live events with a static reference table
- `foreachBatch` — custom sink logic with batch-mode DataFrame API
- Multi-query orchestration: running two streaming queries in parallel
- Schema evolution considerations in streaming pipelines

**Exam relevance:** Streaming (10%) — `foreachBatch`, stream-static joins, and multi-query patterns appear on the exam.

In [ ]:
from pyspark.sql.functions import (
    col, to_timestamp, window, count, sum, avg, round,
    current_timestamp, broadcast, when, coalesce, lit
)
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, TimestampType
import tempfile, os, time, json

LAKE     = tempfile.mkdtemp(prefix="streaming_lab_")
BRONZE   = os.path.join(LAKE, "bronze")
SILVER   = os.path.join(LAKE, "silver")
GOLD     = os.path.join(LAKE, "gold")
CKPT     = os.path.join(LAKE, "checkpoints")
SRC      = os.path.join(LAKE, "source")

for d in [BRONZE, SILVER, GOLD, CKPT, SRC]:
    os.makedirs(d, exist_ok=True)

print(f"Lake: {LAKE}")

## 1. Seed Source Data

In [ ]:
raw_schema = StructType([
    StructField("order_id",   StringType(),  True),
    StructField("customer",   StringType(),  True),
    StructField("product_id", StringType(),  True),
    StructField("qty",        IntegerType(), True),
    StructField("unit_price", IntegerType(), True),
    StructField("event_time", TimestampType(), True),
])

seed = [
    ("ORD-001", "Alice",  "PROD-A", 2, 50, "2024-06-01 10:01:00"),
    ("ORD-002", "Bob",    "PROD-B", 1, 120,"2024-06-01 10:02:00"),
    ("ORD-003", "Carol",  "PROD-A", 3, 50, "2024-06-01 10:03:00"),
    ("ORD-004", "Alice",  "PROD-C", 1, 200,"2024-06-01 10:07:00"),
    ("ORD-005", "Dave",   "PROD-B", 2, 120,"2024-06-01 10:08:00"),
    # Duplicate
    ("ORD-002", "Bob",    "PROD-B", 1, 120,"2024-06-01 10:02:00"),
    # Bad row (null customer)
    ("ORD-006", None,     "PROD-A", 1, 50, "2024-06-01 10:09:00"),
]

seed_df = spark.createDataFrame(
    [(r[0], r[1], r[2], r[3], r[4], to_timestamp(lit(r[5]), "yyyy-MM-dd HH:mm:ss")) for r in seed],
    schema=raw_schema
)
seed_df.write.mode("overwrite").json(SRC)
print(f"Source rows: {seed_df.count()}")

## 2. Static Reference Table

In [ ]:
# Product catalogue — small, loaded once per micro-batch
products = spark.createDataFrame([
    ("PROD-A", "Laptop Stand",  "Accessories"),
    ("PROD-B", "Mechanical KB", "Electronics"),
    ("PROD-C", "Monitor 4K",    "Electronics"),
], ["product_id", "product_name", "category"])

products.show()

## 3. Stream → Bronze with foreachBatch

`foreachBatch` gives you access to each micro-batch as a regular (batch) DataFrame.

This lets you:
- Upsert (merge) into Delta tables
- Write to multiple destinations
- Apply batch-only operations (e.g., JDBC upsert)

In [ ]:
raw_stream = spark.readStream \
    .schema(raw_schema) \
    .json(SRC)

def write_bronze(batch_df, batch_id):
    print(f"[Bronze] batch_id={batch_id}, rows={batch_df.count()}")
    batch_df.withColumn("_batch_id", lit(batch_id)) \
            .write.mode("append").parquet(BRONZE)

q_bronze = raw_stream.writeStream \
    .foreachBatch(write_bronze) \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CKPT, "bronze")) \
    .start()

q_bronze.awaitTermination(timeout=60)
print(f"Bronze rows: {spark.read.parquet(BRONZE).count()}")

## 4. Stream + Static Join (Bronze → Silver)

In [ ]:
bronze_stream = spark.readStream \
    .schema(raw_schema.add("_batch_id", IntegerType())) \
    .parquet(BRONZE)

def write_silver(batch_df, batch_id):
    # Enrichment: join with static product catalogue
    enriched = batch_df.join(broadcast(products), on="product_id", how="left")

    # Quality: drop nulls, dedup, compute total_amount
    from pyspark.sql.functions import expr
    clean = enriched \
        .filter(col("customer").isNotNull()) \
        .dropDuplicates(["order_id"]) \
        .withColumn("total_amount", col("qty") * col("unit_price"))

    print(f"[Silver] batch_id={batch_id}, rows={clean.count()}")
    clean.write.mode("append").parquet(SILVER)

q_silver = bronze_stream.writeStream \
    .foreachBatch(write_silver) \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CKPT, "silver")) \
    .start()

q_silver.awaitTermination(timeout=60)
print("Silver rows:")
spark.read.parquet(SILVER).show(truncate=False)

## 5. Windowed Aggregation → Gold

In [ ]:
silver_schema = StructType([
    StructField("order_id",     StringType(),  True),
    StructField("customer",     StringType(),  True),
    StructField("product_id",   StringType(),  True),
    StructField("qty",          IntegerType(), True),
    StructField("unit_price",   IntegerType(), True),
    StructField("event_time",   TimestampType(), True),
    StructField("_batch_id",    IntegerType(), True),
    StructField("product_name", StringType(),  True),
    StructField("category",     StringType(),  True),
    StructField("total_amount", IntegerType(), True),
])

silver_stream = spark.readStream.schema(silver_schema).parquet(SILVER)

def write_gold(batch_df, batch_id):
    agg = batch_df.groupBy("category") \
        .agg(
            count("order_id").alias("orders"),
            sum("total_amount").alias("revenue"),
        )
    print(f"[Gold] batch_id={batch_id}")
    agg.show()
    agg.write.mode("overwrite").parquet(os.path.join(GOLD, f"batch_{batch_id}"))

q_gold = silver_stream.writeStream \
    .foreachBatch(write_gold) \
    .trigger(availableNow=True) \
    .option("checkpointLocation", os.path.join(CKPT, "gold")) \
    .start()

q_gold.awaitTermination(timeout=60)
print("Gold written.")

## 6. Managing Multiple Streaming Queries

In [ ]:
# List all active streaming queries
active = spark.streams.active
print(f"Active queries: {len(active)}")
for q in active:
    print(f"  - {q.name or q.id}: isActive={q.isActive}")

# Stop all
for q in spark.streams.active:
    q.stop()
print("All streams stopped.")

## 7. Stream-Static Join Rules

| Join type | Stream side | Static side | Supported? |
|---|---|---|---|
| Inner | Left | Right | ✅ |
| Left outer | Left | Right | ✅ |
| Right outer | Left | Right | ❌ |
| Inner | Right | Left | ✅ |
| Stream-stream | Both | — | ✅ (with watermark) |

> **Exam tip:** The static side is re-read on every micro-batch. If you cache the static DataFrame before the stream starts, Spark reuses the cached version.

---

## Day 21 Checklist

- [ ] Used `foreachBatch` to write each micro-batch with custom logic
- [ ] Joined a streaming DataFrame with a static reference table (broadcast)
- [ ] Ran Bronze → Silver → Gold pipeline as chained streaming queries
- [ ] Listed and stopped active streaming queries with `spark.streams.active`
- [ ] Know that static side in stream-static join is re-read each micro-batch

---

## Week 3 Complete!

| Day | Topic | Exam Domain |
|---|---|---|
| 15 | Batch Pipeline (Medallion ETL) | Architecture 20% + DataFrame API 30% |
| 16 | Spark UI & Troubleshooting | Architecture 20% + Optimisation 10% |
| 17 | Memory Management & Accumulators | Architecture 20% + Optimisation 10% |
| 18 | Pandas API on Spark | Pandas API 5% |
| 19 | Structured Streaming intro | Streaming 10% |
| 20 | Streaming Aggregations & Watermarks | Streaming 10% |
| 21 | Streaming Lab | Streaming 10% |

**Next:** Week 4 — Optimisation, Delta Lake, Spark Connect, Certification Practice